In [1]:
import pandas as pd
import numpy as np
import os

# --- 1. 定义文件路径 ---
# 因为你的 notebook 在 'notebooks' 文件夹中，所以我们需要使用 '..' 来返回上一级 (项目根目录)

# 原始数据路径
raw_file_path = '../data/raw/kaggle_data/PRSA_Data_Aotizhongxin_20130301-20170228.csv'

# 处理后数据的保存路径
processed_dir = '../data/processed/'
processed_file_path = os.path.join(processed_dir, 'cleaned_PRSA_Data.csv')

# 确保 'processed' 文件夹存在
os.makedirs(processed_dir, exist_ok=True)

print(f"将从这里读取数据: {raw_file_path}")
print(f"将保存数据到: {processed_file_path}")

# --- 2. 加载数据 ---
try:
    df = pd.read_csv(raw_file_path)
    print("原始数据加载成功。")
    print(df.info())
    print("\n原始数据缺失值统计:")
    print(df.isnull().sum())
except FileNotFoundError:
    print(f"错误：找不到文件，请确认路径 '{raw_file_path}' 是否正确。")
    # 如果出错，后续代码将不会执行
    raise

# --- 3. 数据清洗与整理 ---

# [步骤 3.1] 合并 'year', 'month', 'day', 'hour' 为 'datetime' 索引
print("\n[处理中] 1/5 - 正在创建 datetime 索引...")
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
df.set_index('datetime', inplace=True)

# [步骤 3.2] 丢弃不必要的列
print("[处理中] 2/5 - 正在丢弃无用列 (No, station, year, month, day, hour)...")
df.drop(['No', 'station', 'year', 'month', 'day', 'hour'], axis=1, inplace=True)

# [步骤 3.3] 填充缺失值 (NaN)
print("[处理中] 3/5 - 正在填充数值型列 (线性插值)...")
# 选择所有数值型列
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].interpolate(method='linear')

print("[处理中] 4/5 - 正在填充 'wd' 列 (前向填充)...")
df['wd'] = df['wd'].ffill()

# 检查是否还有缺失值（此时 'wd' 可能还有 NaN，如果它在第一行就是 NaN）
# 为了保险起见，对 ffill 后可能仍然存在的 NaN，用 bfill (后向填充)
df['wd'] = df['wd'].bfill() 

# [步骤 3.4] 对类别型数据 'wd' (风向) 进行 One-Hot 编码
print("[处理中] 5/5 - 正在对 'wd' 列进行 One-Hot 编码...")
df = pd.get_dummies(df, columns=['wd'], prefix='wd')

# --- 4. 保存处理后的数据 ---
df.to_csv(processed_file_path)

print("\n--- 数据清理完成! ---")
print(f"已成功保存清理后的数据到: {processed_file_path}")
print("\n清理后数据的信息:")
print(df.info())
print("\n清理后数据的前5行:")
print(df.head())

将从这里读取数据: ../data/raw/kaggle_data/PRSA_Data_Aotizhongxin_20130301-20170228.csv
将保存数据到: ../data/processed/cleaned_PRSA_Data.csv
原始数据加载成功。
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   No       35064 non-null  int64  
 1   year     35064 non-null  int64  
 2   month    35064 non-null  int64  
 3   day      35064 non-null  int64  
 4   hour     35064 non-null  int64  
 5   PM2.5    34139 non-null  float64
 6   PM10     34346 non-null  float64
 7   SO2      34129 non-null  float64
 8   NO2      34041 non-null  float64
 9   CO       33288 non-null  float64
 10  O3       33345 non-null  float64
 11  TEMP     35044 non-null  float64
 12  PRES     35044 non-null  float64
 13  DEWP     35044 non-null  float64
 14  RAIN     35044 non-null  float64
 15  wd       34983 non-null  object 
 16  WSPM     35050 non-null  float64
 17  station  35064 non-null  ob